## Node Prediction with the Simple API

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/distributed_graph_flow/blob/main/doc/docs/tutorial/node_prediction.ipynb)

Node prediction GNNs use supervised learning to predict node feature values
(categorical labels for classification, and numerical labels for regression).

**Part 1:** This tutorial shows how to train, analyze, and evaluate a model to
predict the label feature in the MAG citation graph. In this graph, authors,
papers, fields of study, and institutions constitute distinct node sets.
Relationships—such as authors writing papers, author affiliations with
institutions, paper topics, and citations between papers—are defined by specific
edge sets. The objective is to predict the domain of a paper stored in the
`labels` feature.

**Part 2:** The first part of the tutorial trains a GNN with access to all
articles and citations. The second part addresses real-world constraints where
data availability depends on the time of publication. This model is trained and
tested using only historical data for each node's prediction.

## Importing libraries

In [1]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import dgf  # Import Graph Flow

## Getting the graph data

We first fetch the MAG graph.

In [2]:
# Download the Mag graph from the OGB repo.
graph, schema = dgf.io.fetch_ogb_graph("mag")

Caching mag graph at /tmp/gf_fetch/mag.cache
OGB dependency not available. Downloading graph from CNS.


Let's look at the graph structure.

In [3]:
# Show the schema
dgf.analyse.print_schema(schema)

Graph Schema:

Node Sets:
  author:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  field_of_study:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  institution:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  paper:
    | Feature   | Format     | Semantic    | Shape   | Num cat. vals   |
    |-----------|------------|-------------|---------|-----------------|
    | #id       | BYTES      | PRIMARY_ID  | None    | None            |
    | #split    | BYTES      | CATEGORICAL | None    | None           

## Part 1: Training a time-agnostic model

We train a model to predict the `labels` feature of the `paper` nodeset.

In [4]:
model = dgf.learning.train_node_model(
    graph=graph,
    schema=schema,
    target_nodeset="paper",
    target_column="labels",
    # Reduce the number of hops and train steps, for the demo to be faster.
    num_sampling_hops=1,
    num_train_steps=5000,
)

Using gpu JAX backend
Graph input schema:
Node Sets:
  author:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  field_of_study:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  institution:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  paper:
    | Feature   | Format     | Semantic    | Shape   | Num cat. vals   |
    |-----------|------------|-------------|---------|-----------------|
    | #id       | BYTES      | PRIMARY_ID  | None    | None            |
    | #split    | BYTES      | CATEGORICAL 

[Warning] No normalizer created for node set 'author', feature '#id'.
[Warning] No normalizer created for node set 'paper', feature '#id'.
[Warning] No normalizer created for node set 'field_of_study', feature '#id'.
[Warning] No normalizer created for node set 'institution', feature '#id'.


Compute graph statistics for padding
  padding: Node Sets:
  author: 256 nodes
  field_of_study: 384 nodes
  institution: 2 nodes
  paper: 542 nodes

Edge Sets:
  affiliated_with: 2 edges
  cites: 507 edges
  has_topic: 384 edges
  writes: 256 edges
Preparing dataset finished in 4.40 seconds
Normalizer:
Graph Normalizer:

Node Sets:
  author:
    (No normalizers)

  field_of_study:
    (No normalizers)

  institution:
    (No normalizers)

  paper:
    - #split: DictionaryIndexNormalizer
    - feat: IdentityNormalizer
    - labels: IdentityNormalizer
    - year: SoftQuantileNormalizer

Edge Sets:
  affiliated_with: (Source: author, Target: institution)
    (No normalizers)

  cites: (Source: paper, Target: paper)
    (No normalizers)

  has_topic: (Source: paper, Target: field_of_study)
    (No normalizers)

  writes: (Source: author, Target: paper)
    (No normalizers)

Normalized graph schema:
Node Sets:
  author:
    (No features)

  field_of_study:
    (No features)

  institution:

Caching validation dataset: 100%|██████████| 1000/1000 [00:06<00:00, 148.22it/s]

Caching validation dataset finished in 6.75 seconds
Number of cache validation batches: 1000
Training model
Generate first batch to initialize model
Create model variables


...

step:8000 train-accuracy:0.2634 train-loss:2.9889 valid-accuracy:0.2637 valid-loss:2.9987


Training:  90%|█████████ | 9013/10000 [02:17<00:38, 25.91it/s, step=9000, train-accuracy=0.2753, train-loss=2.9800, valid-accuracy=0.2637, valid-loss=2.9987]

step:9000 train-accuracy:0.2753 train-loss:2.9800 valid-accuracy:0.2680 valid-loss:2.9804


Training: 100%|██████████| 10000/10000 [02:28<00:00, 67.42it/s, step=9900, train-accuracy=0.2691, train-loss=2.9823, valid-accuracy=0.2680, valid-loss=2.9804]


step:10000 train-accuracy:0.2691 train-loss:2.9823 valid-accuracy:0.2672 valid-loss:2.9761
Final metrics: {'step': '9900', 'train-accuracy': '0.2691', 'train-loss': '2.9823', 'valid-accuracy': '0.2680', 'valid-loss': '2.9804'}
Training model finished in 170.22 seconds


Once trained, it is important to look at the model:

In [5]:
model.describe()

After training, a model is generally evaluated on a test dataset/graph.

**Note:** We don't have a test graph, so we use our training dataset here. In a
real pipeline, evaluating a model on a training dataset make little sense.

In [6]:
model.evaluate(graph)

Evaluating model on 10000 samples


Inference:   0%|          | 0/625 [00:00<?, ?it/s]

...Tracing model


Inference: 100%|██████████| 625/625 [00:05<00:00, 115.33it/s]


Evaluation(loss=None, accuracy=0.2723, rmse=None, r2=None, num_examples=10000, num_examples_weighted=None, mrr=None, auc=None, hit_at={}, user_metrics={})

We can make predictions for individual nodes:

In [7]:
# Predict the probability of each label class for the nodes 0 and 1.
predictions = model.predict(graph, seed_node_idxs=[0, 1])
print("\npredictions's shape [node idx, class idx]:", predictions.shape)

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

...Tracing model


Inference: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it]


predictions's shape [node idx, class idx]: (2, 349)


## Part 2: Training a time-aware model

The graph schema shows (see `dgf.analyse.print_schema()`) that the `paper`
nodeset has a `year` feature; this is the year of publication of the paper. In
this second part, we want to train and evaluate the model as if the GNN model
were applied at the time of publication, that is, it did not have access to
information about papers published after. This is called time-aware modeling.

For time-aware modeling, we need a timestamp on the target nodeset
(`paper` in our case) and on the edgesets we want to filter (`writes`, `cites`
and `has_topic`; but not `affiliated_with`). In this MAG dataset, it makes sense
to simply propagate the `paper` nodeset timestamp to those edgesets: A paper is
written at the time of its publication.

The method `dgf.transform.propagate_timestamp_to_edges` does this automatically.

In addition, we want the `paper`'s `year` feature to also have a `TIMESTAMP` semantic.

In [8]:
time_graph, time_schema = dgf.transform.propagate_timestamp_to_edges(
    graph=graph,
    schema=schema,
    target_edgesets=["has_topic", "cites", "writes"],
    node_timestamps={"paper": "year"},
    target_feature="year",  # Name of the new edge features.
)

time_schema.node_sets["paper"].features[
    "year"
].semantic = dgf.data.FeatureSemantic.TIMESTAMP

dgf.analyse.print_schema(time_schema)

Graph Schema:

Node Sets:
  author:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  field_of_study:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  institution:
    | Feature   | Format   | Semantic   | Shape   | Num cat. vals   |
    |-----------|----------|------------|---------|-----------------|
    | #id       | BYTES    | PRIMARY_ID | None    | None            |

  paper:
    | Feature   | Format     | Semantic    | Shape   | Num cat. vals   |
    |-----------|------------|-------------|---------|-----------------|
    | #id       | BYTES      | PRIMARY_ID  | None    | None            |
    | #split    | BYTES      | CATEGORICAL | None    | None           

We can train the model with `time_aware=True`.

In [9]:
model = dgf.learning.train_node_model(
    graph=time_graph,
    schema=time_schema,
    target_nodeset="paper",
    target_column="labels",
    time_aware=True,
    verbose=1,  # Print less logs than before.
    # Reduce the number of hops and train steps, for the demo to be faster.
    num_sampling_hops=1,
    num_train_steps=5000,
)

Preparing dataset
Num. training seed nodes: 704389, Num. validation seed nodes: 32000


[Warning] No normalizer created for node set 'author', feature '#id'.
[Warning] No normalizer created for node set 'paper', feature '#id'.
[Warning] No normalizer created for node set 'paper', feature 'year'.
[Warning] No normalizer created for node set 'field_of_study', feature '#id'.
[Warning] No normalizer created for node set 'institution', feature '#id'.


Preparing dataset finished in 4.46 seconds
Caching validation dataset
Caching validation dataset finished in 4.06 seconds
Number of cache validation batches: 1000
Training model
Generate first batch to initialize model
Create model variables
...Tracing model
Create model variables finished in 15.80 seconds
Will validate model every 1000 step(s)
Will checkpoint model every 1000 step(s)
Start training. The first two steps are generally slow.


Training:   0%|          | 0/10000 [00:00<?, ?it/s]

...Tracing model


...

...Tracing model


Training:  10%|█         | 1015/10000 [00:29<11:08, 13.44it/s, step=1000, train-accuracy=0.1894, train-loss=3.5854]

Validation loop took 2.70s (only printed once)


Training: 100%|██████████| 10000/10000 [02:21<00:00, 70.86it/s, step=9900, train-accuracy=0.2644, train-loss=2.9738, valid-accuracy=0.2657, valid-loss=2.9960]


Final metrics: {'step': '9900', 'train-accuracy': '0.2644', 'train-loss': '2.9738', 'valid-accuracy': '0.2657', 'valid-loss': '2.9960'}
Training model finished in 158.20 seconds


We look at the model:

In [10]:
model.describe()

You can see the effect of time-aware sampling in two places:

- In the "graph sampling" tab, the `edgeset_timestamp_features` field show the edge features to time-filter on.
- In the "padding tab", the paper nodeset padding is smaller than before. Because of filtering, each paper node has in average half the number of edges, and the graph samples (used internally to train the GNN) are smaller.

The model quality is only a little reduced. In this dataset, future information is not critical.
